# (1) Import libraries

> **Chuyển đổi từ `0. Getting Started/Iris_Bayesian_decision_making.ipynb` sang PyTorch**


In [1]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(42)
torch.manual_seed(42)


# (2) Preparing Iris dataset


In [2]:
try:
    from sklearn import datasets
    iris = datasets.load_iris()
    X_all, y_all = iris.data.astype(np.float32), iris.target.astype(np.int64)
except ImportError:
    c0 = np.random.multivariate_normal([5.0, 3.4, 1.5, 0.2], np.diag([0.12, 0.14, 0.03, 0.01]), 50)
    c1 = np.random.multivariate_normal([5.9, 2.7, 4.2, 1.3], np.diag([0.26, 0.09, 0.22, 0.04]), 50)
    c2 = np.random.multivariate_normal([6.5, 2.9, 5.5, 2.0], np.diag([0.40, 0.10, 0.30, 0.07]), 50)
    X_all = np.vstack([c0, c1, c2]).astype(np.float32)
    y_all = np.array([0]*50 + [1]*50 + [2]*50, dtype=np.int64)

train_indx = np.random.choice(range(0, 150), int(y_all.shape[0] * 0.6), replace=False)
test_indx = np.array([indx for indx in np.arange(0, 150) if indx not in train_indx])

x_train, y_train = X_all[train_indx], y_all[train_indx]
x_test, y_test = X_all[test_indx], y_all[test_indx]
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)


(90, 4) (90,) (60, 4) (60,)


# (3) Parameters estimation (MLE)


In [3]:
means, covs, priors = [], [], []
for c in range(3):
    xc = x_train[y_train == c]
    mu_c = np.mean(xc, axis=0)
    sigma_c = np.cov(xc, rowvar=False) + np.eye(4) * 1e-4
    means.append(mu_c)
    covs.append(sigma_c)
    priors.append(len(xc) / len(x_train))
    print(f'Class {c} mean:', np.round(mu_c, 3))


Class 0 mean: [4.951 3.338 1.458 0.223]
Class 1 mean: [5.979 2.77  4.24  1.294]
Class 2 mean: [6.516 2.928 5.467 2.031]


# (4) Bayesian decision making & PyTorch Softmax Classifier


In [4]:
# 4.1. Gaussian Bayesian Decision Rule bằng PyTorch Tensor
def gaussian_log_posterior(x_tensor, means, covs, priors):
    scores = []
    for c in range(len(means)):
        mu = torch.tensor(means[c], dtype=torch.float32)
        cov = torch.tensor(covs[c], dtype=torch.float32)
        dist = torch.distributions.MultivariateNormal(mu, cov)
        scores.append(dist.log_prob(x_tensor) + np.log(priors[c]))
    return torch.stack(scores, dim=1)

x_test_t = torch.tensor(x_test)
y_test_t = torch.tensor(y_test)
bayes_pred = torch.argmax(gaussian_log_posterior(x_test_t, means, covs, priors), dim=1)
print('PyTorch Bayesian classification accuracy: %.4f' % (bayes_pred == y_test_t).float().mean().item())

# 4.2. PyTorch Linear Softmax Classifier (nn.Linear + CrossEntropyLoss)
model = nn.Linear(4, 3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)
x_train_t = torch.tensor(x_train)
y_train_t = torch.tensor(y_train)

for epoch in range(200):
    optimizer.zero_grad()
    loss = criterion(model(x_train_t), y_train_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    nn_pred = torch.argmax(model(x_test_t), dim=1)
    print('PyTorch Softmax Classifier accuracy: %.4f' % (nn_pred == y_test_t).float().mean().item())


PyTorch Bayesian classification accuracy: 0.9500


PyTorch Softmax Classifier accuracy: 0.9000
